In [1]:
# %% Setup
import os
os.environ.setdefault("KMP_DUPLICATE_LIB_OK", "TRUE")  # avoid libomp double-init abort (numpy and torch both bundle OpenMP on macOS)
os.environ.setdefault("OMP_NUM_THREADS", "1")           # avoids an OMP pthread_mutex_init segfault during inference on macOS

import numpy as np
import pandas as pd
import h5py
import torch
from chronos import Chronos2Pipeline
from chronos.chronos2.preprocess import from_data_frame

from cstr_data import load_cstr_data, STATE_COLUMNS, EXOG_COLUMNS

device = "cuda" if torch.cuda.is_available() else "cpu"
pipeline = Chronos2Pipeline.from_pretrained("amazon/chronos-2", device_map=device)
print(f"Loaded Chronos-2 on {device}")

C:\Users\USER-PC\OneDrive\Documents\Masters\Masters-Forecasting\envs\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm



Loading weights:   0%|          | 0/170 [00:00<?, ?it/s]


Loading weights:  18%|█▊        | 31/170 [00:00<00:00, 308.24it/s]


Loading weights:  44%|████▍     | 75/170 [00:00<00:00, 383.56it/s]


Loading weights:  67%|██████▋   | 114/170 [00:00<00:00, 375.09it/s]


Loading weights:  89%|████████▉ | 152/170 [00:00<00:00, 359.09it/s]


Loading weights: 100%|██████████| 170/170 [00:00<00:00, 370.12it/s]

Loaded Chronos-2 on cuda


In [2]:
# %% Load the fine-tuning simulation (separate 5M-point dataset, generated for this purpose) and build training inputs
FT_DOWNSAMPLE = 20
CONTEXT_LENGTH = pipeline.model_context_length        # same cap Chronos2-Forecast.ipynb uses
PREDICTION_LENGTH = pipeline.model_prediction_length   # same cap Chronos2-Forecast.ipynb uses

train_df, _ = load_cstr_data(
    downsample=FT_DOWNSAMPLE,
    input_file="CSTR_InputVectors_FineTune.h5",
    sim_file="CSTR_SimulationData_FineTune_ds20.h5",
)
# drop the *_active flag columns - they aren't passed as covariates at inference time either (see Chronos2-Forecast.ipynb)
train_df = train_df[["item_id", "timestamp"] + list(STATE_COLUMNS.values()) + list(EXOG_COLUMNS.values())]

VAL_FRACTION = 0.1   # time-based split, held out from the tail of the fine-tuning series
split = int(len(train_df) * (1 - VAL_FRACTION))
fit_df, val_df = train_df.iloc[:split].copy(), train_df.iloc[split:].copy()

fit_kwargs = dict(
    target_columns=list(STATE_COLUMNS.values()),
    prediction_length=PREDICTION_LENGTH,
    known_covariates_names=list(EXOG_COLUMNS.values()),
    id_column="item_id",
    timestamp_column="timestamp",
)
fit_inputs = from_data_frame(fit_df, **fit_kwargs)
val_inputs = from_data_frame(val_df, **fit_kwargs)
print(f"Fine-tuning series: {len(fit_df)} points, validation series: {len(val_df)} points")

Fine-tuning series: 22500 points, validation series: 2500 points


In [ ]:
# %% Fine-tune (LoRA) - retries with a smaller batch size on CUDA OOM since this runs unattended
FINETUNE_MODE = "lora"
LEARNING_RATE = 1e-5      # docs recommend a higher LR than the 1e-6 full-finetune default when using LoRA
NUM_STEPS = 2000
BATCH_SIZE = 128
OUTPUT_DIR = "chronos-2-finetuned"

batch_size = BATCH_SIZE
finetuned_pipeline = None
while finetuned_pipeline is None:
    try:
        finetuned_pipeline = pipeline.fit(
            inputs=fit_inputs,
            prediction_length=PREDICTION_LENGTH,
            validation_inputs=val_inputs,
            finetune_mode=FINETUNE_MODE,
            context_length=CONTEXT_LENGTH,
            learning_rate=LEARNING_RATE,
            num_steps=NUM_STEPS,
            batch_size=batch_size,
            output_dir=OUTPUT_DIR,
            finetuned_ckpt_name="finetuned-ckpt",
        )
    except torch.cuda.OutOfMemoryError:
        torch.cuda.empty_cache()
        if batch_size <= 8:
            raise
        batch_size //= 2
        print(f"CUDA OOM - retrying with batch_size={batch_size}")

print(f"Fine-tuning complete (batch_size used: {batch_size})")

CUDA OOM - retrying with batch_size=64


CUDA OOM - retrying with batch_size=32


Step,Training Loss,Validation Loss
100,2.190251,2.959915
200,1.807287,2.774512
300,1.489532,2.638298
400,1.444254,2.534097
500,1.402500,2.465385
600,1.308143,2.403259
700,1.234229,2.391115
800,1.156972,2.360033
900,1.201901,2.335758
1000,1.087761,2.331820


Fine-tuning complete (batch_size used: 32)


In [4]:
# %% Save the fine-tuned model for reuse (loaded by Chronos2-Compare.ipynb)
SAVE_DIR = "chronos-2-finetuned-final"
finetuned_pipeline.save_pretrained(SAVE_DIR)
print(f"Saved fine-tuned pipeline to {SAVE_DIR}")

Saved fine-tuned pipeline to chronos-2-finetuned-final
